# Feature store served online, step by step

A hands-on lab. Run one cell at a time. You will create raw events, materialize features into a fast table, and measure sub-millisecond lookup latency for inference.

**Before you start:** finish `00-setup/`.

## Setup — connect

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, _env

SCHEMA = _env('LB_SCHEMA', 'factory')
EVENTS_TABLE = 'account_events_for_features'
ONLINE_TABLE = 'machine_features_online'

print('Ready.')


## Step 1 — Create raw events table

```sql
CREATE TABLE account_events_for_features (
    event_id BIGSERIAL PRIMARY KEY,
    account_id TEXT NOT NULL,
    event_ts TIMESTAMPTZ DEFAULT now(),
    event_type TEXT,
    cycles INT,
    temp_c NUMERIC(6,2),
    anomaly_score NUMERIC(6,2)
);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {SCHEMA}.{EVENTS_TABLE}")
        cur.execute(f"""
            CREATE TABLE {SCHEMA}.{EVENTS_TABLE} (
                event_id BIGSERIAL PRIMARY KEY,
                account_id TEXT NOT NULL,
                event_ts TIMESTAMPTZ DEFAULT now(),
                event_type TEXT,
                cycles INT,
                temp_c NUMERIC(6,2),
                anomaly_score NUMERIC(6,2)
            )
        """)
        print(f'✓ Created raw events table')


## Step 2 — Seed raw events (10k events across 100 machines)

```sql
INSERT INTO account_events_for_features(...)
SELECT ... FROM generate_series(1, 10000) g
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {SCHEMA}.{EVENTS_TABLE}(account_id, event_ts, event_type, cycles, temp_c, anomaly_score)
            SELECT
                'M-' || (g % 100),
                now() - ((g / 100) || ' days')::interval,
                (ARRAY['run', 'failure', 'maintenance'])[1 + (g % 3)],
                g * 100,
                50.0 + (g % 30),
                2.0 + (g % 5)::numeric / 10
            FROM generate_series(1, 10000) g
        """)
        print(f'✓ Seeded 10k events across 100 machines')


## Step 3 — Create online feature table

```sql
CREATE TABLE machine_features_online (
    account_id TEXT PRIMARY KEY,
    runs_this_month INT,
    failures_this_month INT,
    avg_temp_c NUMERIC(8,2),
    avg_vibration_mm_s NUMERIC(8,2),
    total_cycles INT,
    computed_at TIMESTAMPTZ
);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {SCHEMA}.{ONLINE_TABLE}")
        cur.execute(f"""
            CREATE TABLE {SCHEMA}.{ONLINE_TABLE} (
                account_id TEXT PRIMARY KEY,
                runs_this_month INT,
                failures_this_month INT,
                avg_temp_c NUMERIC(8,2),
                avg_vibration_mm_s NUMERIC(8,2),
                total_cycles INT,
                computed_at TIMESTAMPTZ
            )
        """)
        print(f'✓ Created online feature table')


## Step 4 — Materialize features from events

```sql
INSERT INTO machine_features_online(...)
SELECT
    e.account_id,
    COUNT(CASE WHEN e.event_type = 'run' THEN 1 END),
    COUNT(CASE WHEN e.event_type = 'failure' THEN 1 END),
    ...
FROM account_events_for_features e
WHERE e.event_ts >= now() - interval '30 days'
GROUP BY e.account_id;
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {SCHEMA}.{ONLINE_TABLE}
            SELECT
                e.account_id,
                COUNT(CASE WHEN e.event_type = 'run' THEN 1 END),
                COUNT(CASE WHEN e.event_type = 'failure' THEN 1 END),
                AVG(e.temp_c),
                AVG(e.anomaly_score),
                SUM(e.cycles),
                now()
            FROM {SCHEMA}.{EVENTS_TABLE} e
            WHERE e.event_ts >= now() - interval '30 days'
            GROUP BY e.account_id
        """)
        print(f'✓ Materialized features from events (offline step)')


## Step 5 — Measure online serving latency

Run 100 primary-key lookups and time each one.

```python
latencies = []
for account_id in range(100):
    t0 = time.perf_counter()
    cur.execute(f"SELECT * FROM machine_features_online WHERE account_id = %s", [f'M-{account_id}'])
    cur.fetchone()
    latencies.append((time.perf_counter() - t0) * 1000)
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        latencies = []
        for account_id in range(100):
            t0 = time.perf_counter()
            cur.execute(f"SELECT * FROM {SCHEMA}.{ONLINE_TABLE} WHERE account_id = %s", [f'M-{account_id % 100}'])
            row = cur.fetchone()
            latencies.append((time.perf_counter() - t0) * 1000)
        
        if latencies:
            avg_ms = sum(latencies) / len(latencies)
            min_ms = min(latencies)
            max_ms = max(latencies)
            print(f'✓ Measured {len(latencies)} lookups:')
            print(f'  Average: {avg_ms:.2f} ms')
            print(f'  Min:     {min_ms:.2f} ms')
            print(f'  Max:     {max_ms:.2f} ms')
            print(f'  Result: single-digit millisecond latency, ready for inference')


## What you learned

Feature stores work by materializing (computing offline) and serving online (fast lookups at prediction time). Materialize aggregates from raw events into a dedicated table, indexed for speed. Serve means primary-key lookups, single-digit-millisecond latency, no computation at prediction time. Same database for both; no sync between systems.

Next: **One query, every type** builds a single table with JSON, vectors, and geospatial data.